    # Week 9 · Python Raster & Remote Sensing
    
    Extend the QGIS raster workflows into Python for reproducible change detection and zonal statistics.
    


    ## Learning goals
    
    - Open and inspect raster datasets with Rasterio/xarray.
    - Clip and reproject rasters to match vector areas of interest.
    - Calculate zonal statistics and simple change metrics.
    


    ## 1. Imports & paths
    


In [ ]:
    from pathlib import Path
    
    import geopandas as gpd
    import rasterio
    from rasterio.mask import mask
    import numpy as np
    
    DATA_ROOT = Path("..") / "data" / "processed" / "week09"
    RASTER_BEFORE = DATA_ROOT / "sentinel_before.tif"
    RASTER_AFTER = DATA_ROOT / "sentinel_after.tif"
    AOI_PATH = DATA_ROOT / "aoi.geojson"
    
    for path in [RASTER_BEFORE, RASTER_AFTER, AOI_PATH]:
        if not path.exists():
            raise FileNotFoundError(f"Missing required dataset: {path}")
    
    aoi = gpd.read_file(AOI_PATH).to_crs(4326)
    


    ## 2. Clip rasters to AOI
    


In [ ]:
    def clip_raster(path, shapes):
        with rasterio.open(path) as src:
            out_image, out_transform = mask(src, shapes.geometry, crop=True)
            out_meta = src.meta.copy()
            out_meta.update({
                "height": out_image.shape[1],
                "width": out_image.shape[2],
                "transform": out_transform,
            })
        return out_image, out_meta
    
    before_clip, before_meta = clip_raster(RASTER_BEFORE, aoi)
    after_clip, after_meta = clip_raster(RASTER_AFTER, aoi)
    


    ## 3. Compute change index
    
    Replace the placeholder calculation with domain-specific band math (e.g., NDVI difference).
    


In [ ]:
    # TODO: Update band indices based on dataset
    before_band = before_clip[0].astype(float)
    after_band = after_clip[0].astype(float)
    change = after_band - before_band
    change_mean = float(np.nanmean(change))
    print(f"Mean change value: {change_mean:.3f}")
    


    ## 4. Zonal statistics
    
    Summarise change values per polygon (e.g., neighbourhood, watershed) to align with QGIS reporting outputs.
    


In [ ]:
    from rasterstats import zonal_stats
    
    zones_path = DATA_ROOT / "zones.geojson"
    if not zones_path.exists():
        raise FileNotFoundError("Provide polygons at data/processed/week09/zones.geojson")
    
    zones = gpd.read_file(zones_path)
    stats = zonal_stats(zones, change, affine=before_meta["transform"], stats=["mean", "max", "min"], nodata=np.nan)
    zones_stats = zones.join(gpd.GeoDataFrame(stats))
    zones_stats.head()
    


    ## 5. Quick visualisation
    


In [ ]:
    import matplotlib.pyplot as plt
    
    fig, ax = plt.subplots(1, 1, figsize=(9, 6))
    zones_stats.plot(column="mean", cmap="coolwarm", legend=True, ax=ax)
    ax.set_title("Average change per zone")
    ax.set_axis_off()
    plt.show()
    


    ## 6. Save outputs
    


In [ ]:
    OUTPUT_DIR = DATA_ROOT / "outputs"
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    zones_stats.to_file(OUTPUT_DIR / "zones_change.gpkg", driver="GPKG", layer="zones_change")
    


    ## 7. Reflection
    


In [ ]:
    reflection = {
        "raster_learning": "What step was most surprising about the raster workflow?",
        "tool_comparison": "When would you stay in QGIS vs Python for raster analysis?",
        "data_gaps": "Which datasets do you need to refine before Week 10?",
    }
    for key, value in reflection.items():
        print(f"{key}: {value}")
    
